# 🔢 Numeric Embedding Evaluation
Evaluate ModernBERT (base or LoRA fine-tuned) on numeric similarity tasks.

**Sections:**
1. Setup & Config
2. Load Model
3. Core Functions (pooling, embed, metrics)
4. Triplet Evaluation
5. Similarity Probing & Test Suites

## 1. Setup & Config

In [1]:
# Install dependencies — comment out after first run
# !pip install -q transformers peft tqdm numpy scikit-learn torch


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:

! ls -lt /content/drive/MyDrive/numeric_finetune_data/trained_model

total 16
drwx------ 2 root root 4096 Apr 15 04:57 modernbert_lora_contrastive_corrected_CDL_GPTl_trainer_3_04142026
drwx------ 2 root root 4096 Apr 15 04:56 modernbert_lora_contrastive_corrected_CDL_GPTl_4132026
drwx------ 2 root root 4096 Apr 12 03:18 modernbert_lora_contrastive_corrected_CDL_GPTl_epoch_2
drwx------ 2 root root 4096 Apr 12 02:22 modernbert_lora_contrastive_corrected_CDL_GPTl_epoch_1


In [7]:
import os

# ── PATHS ─────────────────────────────────────────────────────────────
WORK_DIR   = '/content/drive/MyDrive/numeric_finetune_data'
MODEL_DIR  = f'{WORK_DIR}/trained_model'
DATA_DIR   = f'{WORK_DIR}/NumerSense'

# ── MODEL ──────────────────────────────────────────────────────────────
BASE_MODEL = 'answerdotai/ModernBERT-base'

# Set USE_LORA = True and fill in LORA_PATH to evaluate a fine-tuned model
USE_LORA   = True
LORA_PATH  = f'{MODEL_DIR}/modernbert_lora_contrastive_corrected_CDL_GPTl_trainer_3_04142026'
#LORA_PATH  = f'{MODEL_DIR}/modernbert_lora_contrastive-corrected2'

#

# ── EVAL SETTINGS ──────────────────────────────────────────────────────
BATCH_SIZE            = 32
MAX_LENGTH            = 128
RECALL_K              = [1, 5, 10]
NUM_SAMPLED_NEGATIVES = 8

# ── DEVICE ─────────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {BASE_MODEL}')
print(f'LoRA   : {USE_LORA}  →  {LORA_PATH if USE_LORA else "(base model)"}')


Device : cuda
Model  : answerdotai/ModernBERT-base
LoRA   : True  →  /content/drive/MyDrive/numeric_finetune_data/trained_model/modernbert_lora_contrastive_corrected_CDL_GPTl_trainer_3_04142026


## 2. Load Model

In [ ]:
del base_model
del model

NameError: name 'base_model' is not defined

In [8]:
USE_LORA = True

from transformers import AutoTokenizer, AutoModel

tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModel.from_pretrained(BASE_MODEL)

if USE_LORA:
    from peft import PeftModel
    model = PeftModel.from_pretrained(base_model, LORA_PATH)
else:
    model = base_model

model = model.to(DEVICE).eval()
print('Model loaded ✓')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded ✓


## 3. Core Functions

### 3a. Pooling & Embedding

In [9]:
import torch.nn.functional as F
from tqdm import tqdm

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

@torch.no_grad()
def embed_texts(texts):
    """Embed a list of strings → L2-normalized numpy array (N, hidden_dim)."""
    all_emb = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding', leave=False):
        batch = texts[i : i + BATCH_SIZE]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
        outputs = model(**inputs)
        emb = mean_pooling(outputs, inputs['attention_mask'])
        emb = F.normalize(emb, p=2, dim=1)
        all_emb.append(emb.cpu())
    import numpy as np
    return torch.cat(all_emb, dim=0).numpy()


### 3b. Metrics

In [10]:
import numpy as np
import random

def triplet_accuracy(A, P, N):
    """Fraction of triplets where sim(a,p) > sim(a,n)."""
    return float(np.mean(np.sum(A*P, axis=1) > np.sum(A*N, axis=1)))

def cosine_gap(A, P, N):
    """Mean sim(a,p) - sim(a,n)."""
    return float(np.mean(np.sum(A*P, axis=1) - np.sum(A*N, axis=1)))

def pairwise_mrr(A, P, N):
    """MRR over {positive, negative} pair only."""
    pos = np.sum(A*P, axis=1)
    neg = np.sum(A*N, axis=1)
    return float(np.mean(1.0 / np.where(pos > neg, 1, 2)))

def recall_at_k_sampled(A, P, N, k, num_extra_negs=8):
    n = len(A)
    hits = []
    for i in range(n):
        neg_idx = random.sample([j for j in range(n) if j != i], min(num_extra_negs, n-1))
        candidates = [P[i], N[i]] + [N[j] for j in neg_idx]
        labels     = [1]   + [0]   * (1 + len(neg_idx))
        sims  = np.dot(candidates, A[i])
        topk  = np.argsort(-sims)[:k]
        hits.append(any(labels[j] == 1 for j in topk))
    return float(np.mean(hits))

def mrr_at_k_sampled(A, P, N, k, num_extra_negs=8):
    n = len(A)
    rr = []
    for i in range(n):
        neg_idx = random.sample([j for j in range(n) if j != i], min(num_extra_negs, n-1))
        candidates = [P[i], N[i]] + [N[j] for j in neg_idx]
        sims  = np.dot(candidates, A[i])
        order = np.argsort(-sims)
        rank  = int(np.where(order == 0)[0][0]) + 1  # index 0 = positive
        rr.append(1.0 / rank if rank <= k else 0.0)
    return float(np.mean(rr))

print('Metric functions defined ✓')


Metric functions defined ✓


### 3c. Data Loading & Evaluation Runner

In [11]:
import json
import pandas as pd

def load_triplets(path):
    anchors, positives, negatives = [], [], []
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            anchors.append(row['comment'])
            positives.append(row['positive_rewritten'])
            negatives.append(row['negative_rewritten'])
    return anchors, positives, negatives

def evaluate(triplet_file, verbose=True):
    """Run full triplet eval on a JSONL file. Returns dict of metric → float."""
    if verbose:
       print(f'\n── {triplet_file.split("/")[-1]} ──')
    anchors, positives, negatives = load_triplets(triplet_file)
    if verbose:
        print(f'  {len(anchors)} triplets loaded')

    A = embed_texts(anchors)
    P = embed_texts(positives)
    N = embed_texts(negatives)

    results = {
        'triplet_accuracy': triplet_accuracy(A, P, N),
        'cosine_gap':       cosine_gap(A, P, N),
        'pairwise_mrr':     pairwise_mrr(A, P, N),
    }
    for k in RECALL_K:
        results[f'recall@{k}'] = recall_at_k_sampled(A, P, N, k, NUM_SAMPLED_NEGATIVES)
        results[f'mrr@{k}']    = mrr_at_k_sampled(A, P, N, k, NUM_SAMPLED_NEGATIVES)

    if verbose:
        for name, val in results.items():
            print(f'  {name:<20s}: {val:.4f}')
    return results

print('Evaluation functions defined ✓')


Evaluation functions defined ✓


## 4. Triplet Evaluation

In [12]:
"""
triplet_accuracy    : 0.8884
  cosine_gap          : 0.0119
  pairwise_mrr        : 0.9442
  recall@1            : 0.7928
  mrr@1               : 0.7855
  recall@5            : 0.9412
  mrr@5               : 0.8599
  recall@10           : 1.0000
  mrr@10              : 0.8666
"""

'\ntriplet_accuracy    : 0.8884\n  cosine_gap          : 0.0119\n  pairwise_mrr        : 0.9442\n  recall@1            : 0.7928\n  mrr@1               : 0.7855\n  recall@5            : 0.9412\n  mrr@5               : 0.8599\n  recall@10           : 1.0000\n  mrr@10              : 0.8666\n'

In [13]:
files = [
    f'{DATA_DIR}/test_general.jsonl',
    f'{DATA_DIR}/test_same.jsonl'
]


all_results = {f.split('/')[-1]: evaluate(f) for f in files}

# Side-by-side comparison table
df = pd.DataFrame(all_results).T.round(4)
print('\n===== COMPARISON =====')
display(df)



── test_general.jsonl ──
  2564 triplets loaded


Embedding:   0%|          | 0/81 [00:00<?, ?it/s]W0415 04:58:37.677000 5328 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


  triplet_accuracy    : 0.9883
  cosine_gap          : 0.1831
  pairwise_mrr        : 0.9941
  recall@1            : 0.8846
  mrr@1               : 0.8857
  recall@5            : 0.9938
  mrr@5               : 0.9274
  recall@10           : 1.0000
  mrr@10              : 0.9323

── test_same.jsonl ──
  6490 triplets loaded


  triplet_accuracy    : 0.9898
  cosine_gap          : 0.1802
  pairwise_mrr        : 0.9949
  recall@1            : 0.9471
  mrr@1               : 0.9421
  recall@5            : 0.9988
  mrr@5               : 0.9707
  recall@10           : 1.0000
  mrr@10              : 0.9696

===== COMPARISON =====


,triplet_accuracy,cosine_gap,pairwise_mrr,recall@1,mrr@1,recall@5,mrr@5,recall@10,mrr@10
test_general.jsonl,0.9883,0.1831,0.9941,0.8846,0.8857,0.9938,0.9274,1.0,0.9323
test_same.jsonl,0.9898,0.1802,0.9949,0.9471,0.9421,0.9988,0.9707,1.0,0.9696


## 5. Similarity Probing & Test Suites

### 5a. Probe Helper

In [14]:
@torch.no_grad()
def embed_single(text):
    """
    Embed a single string, returns a (1, D) tensor on CPU.
    """

    inputs = {k: v.to(DEVICE) for k, v in tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=MAX_LENGTH, padding=True).items()}
    outputs = model(**inputs)
    emb = mean_pooling(outputs, inputs['attention_mask'])
    return F.normalize(emb, p=2, dim=1).cpu()

def probe(anchor, candidates, show_correlation=True):
    """
    Print similarity ranking for anchor vs candidates.
    Optionally compute numeric distance correlation (assumes '<number> <unit>' format).
    """
    anchor_emb = embed_single(anchor)
    rows = []
    for text in candidates:
        emb  = embed_single(text)
        sim  = torch.matmul(anchor_emb, emb.T).item()
        rows.append({'text': text, 'sim': sim, 'tokens': tokenizer.tokenize(text)})

    rows.sort(key=lambda x: x['sim'], reverse=True)

    print(f'\nAnchor: {anchor} ')
    print('Similarity ranking (highest → lowest):')
    for r in rows:
        print(f"  {r['text']:>10} : {r['sim']:.4f} ")
        #print(f"  {r['text']:>10} : {r['sim']:.4f}   ← {r['tokens']}")

    if show_correlation:
        try:
            anchor_val = float(anchor.split()[0])
            dists = [abs(float(r['text'].split()[0]) - anchor_val) for r in rows]
            sims  = [r['sim'] for r in rows]
            corr  = np.corrcoef(dists, sims)[0, 1]
            print(f'\n  Numeric distance correlation: {corr:.4f}')
            print('  (≤ -0.7 = learned numeric ordering | ≈ 0 = no magnitude structure)')
        except (ValueError, IndexError):
            pass
    return rows


### 5b. Built-in Test Suites

In [15]:
TEST_SUITES = {
    "general 1": {
        "anchor" : "13 ml",
        "candidates" : [
    "10 ml", "15 ml", "20 ml", "25 ml",
    "30 ml", "45 ml", "100 ml", "123 ml", "125 ml", "321 ml", "251 ml",
    "252 ml", "259 ml", "257 ml","327 ml","323 ml",
    "32 ml"],
        'description': 'Does the model weight first-digit matches too heavily?',
    },
    "general 2": {
        "anchor" : "321 ml",
        "candidates" : ["321 ml","323 ml","327 ml","252 ml","123 ml","257 ml","259 ml","32 ml","251 ml","30 ml","25 ml","125 ml","20 ml","15 ml","45 ml","100 ml","10 ml"],
        'description': 'Does the model weight first-digit matches too heavily?',
    },
    'Suite 1 — Left-to-Right Digit Bias': {
        'anchor': '321 ml',
        'candidates': ['321 ml', '320 ml', '329 ml', '312 ml',
                       '301 ml', '231 ml', '132 ml', '123 ml', '221 ml', '421 ml'],
        'description': 'Does the model weight first-digit matches too heavily?',
    },
    'Suite 2 — Monotonic Magnitude Decay': {
        'anchor': '500 ml',
        'candidates': ['499 ml', '501 ml', '490 ml', '510 ml',
                       '450 ml', '550 ml', '400 ml', '600 ml', '300 ml', '700 ml', '100 ml', '900 ml'],
        'description': 'Does similarity decay as numeric distance grows?',
    },
    'Suite 3 — Place Value Sensitivity': {
        'anchor': '345 ml',
        'candidates': ['346 ml', '355 ml', '445 ml', '344 ml', '335 ml', '245 ml'],
        'description': 'Is a ±1 change scored closer than ±10 or ±100?',
    },
    'Suite 4 — Prefix vs True Magnitude': {
        'anchor': '199 ml',
        'candidates': ['200 ml', '198 ml', '190 ml', '299 ml', '109 ml', '119 ml', '999 ml'],
        'description': 'Does the true nearest neighbor beat digit-sharing distractors?',
    },
    'Suite 5 — Digit Permutation Trap': {
        'anchor': '247 ml',
        'candidates': ['247 ml', '274 ml', '427 ml', '724 ml',
                       '742 ml', '472 ml', '200 ml', '250 ml', '300 ml'],
        'description': 'Does the model confuse permuted digits with numeric proximity?',
    },
}

# Run all suites
for name, suite in TEST_SUITES.items():
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'  {suite["description"]}')
    probe(suite['anchor'], suite['candidates'])



general 1
  Does the model weight first-digit matches too heavily?

Anchor: 13 ml 
Similarity ranking (highest → lowest):
       15 ml : 0.9828 
       10 ml : 0.9757 
       20 ml : 0.9654 
       25 ml : 0.9594 
       30 ml : 0.9515 
       32 ml : 0.9451 
       45 ml : 0.9288 
      100 ml : 0.9181 
      123 ml : 0.9133 
      125 ml : 0.9080 
      327 ml : 0.8850 
      259 ml : 0.8792 
      321 ml : 0.8786 
      252 ml : 0.8758 
      257 ml : 0.8724 
      251 ml : 0.8693 
      323 ml : 0.8653 

  Numeric distance correlation: -0.9360
  (≤ -0.7 = learned numeric ordering | ≈ 0 = no magnitude structure)

general 2
  Does the model weight first-digit matches too heavily?

Anchor: 321 ml 
Similarity ranking (highest → lowest):
      321 ml : 1.0000 
      323 ml : 0.9834 
      327 ml : 0.9748 
      257 ml : 0.9573 
      259 ml : 0.9542 
      251 ml : 0.9529 
      252 ml : 0.9502 
      123 ml : 0.9468 
      125 ml : 0.9344 
      100 ml : 0.9160 
       45 ml : 0.9149 

### 5c. Custom Probe
_Edit anchor and candidates to test any values you like._

In [16]:
TEST_SUITES = {

    # ──────────────────────────────────────────
    # Suite 1 — Immediate Neighbor Recognition
    # Tests: can model rank ±1 above ±10 above ±100
    # ──────────────────────────────────────────
    "Suite 01 — Immediate Neighbor Recognition (small)": {
        "anchor": "13 ml",
        "candidates": [
            "14 ml", "12 ml",       # ±1
            "15 ml", "11 ml",       # ±2
            "20 ml", "10 ml",       # ±7
            "25 ml", "5 ml",        # ±12
            "30 ml", "3 ml",        # ±17
            "45 ml", "100 ml",      # far
            "321 ml",               # very far
        ],
        "expected_order": ["14 ml", "12 ml", "15 ml", "11 ml", "20 ml", "10 ml",
                           "25 ml", "5 ml", "30 ml", "3 ml", "45 ml", "100 ml", "321 ml"],
        "description": "Immediate neighbors (±1) should rank above distant values",
        "metric": "monotonic",
    },

    "Suite 01b — Immediate Neighbor Recognition (large)": {
        "anchor": "1000 units",
        "candidates": [
            "1001 units", "999 units",
            "1005 units", "995 units",
            "1010 units", "990 units",
            "1050 units", "950 units",
            "1100 units", "900 units",
            "2000 units", "500 units",
        ],
        "expected_order": ["1001 units", "999 units", "1005 units", "995 units",
                           "1010 units", "990 units", "1050 units", "950 units",
                           "1100 units", "900 units", "2000 units", "500 units"],
        "description": "±1 neighbors rank above ±100 rank above order-of-magnitude jumps",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 2 — Monotonic Magnitude Decay
    # Tests: does similarity decay as distance grows?
    # ──────────────────────────────────────────
    "Suite 02 — Monotonic Magnitude Decay (500)": {
        "anchor": "500 ml",
        "candidates": [
            "499 ml", "501 ml",     # ±1
            "490 ml", "510 ml",     # ±10
            "450 ml", "550 ml",     # ±50
            "400 ml", "600 ml",     # ±100
            "300 ml", "700 ml",     # ±200
            "100 ml", "900 ml",     # ±400
        ],
        "expected_order": ["499 ml", "501 ml", "490 ml", "510 ml", "450 ml",
                           "550 ml", "400 ml", "600 ml", "300 ml", "700 ml",
                           "100 ml", "900 ml"],
        "description": "Similarity should decay monotonically as numeric distance grows from 500",
        "metric": "monotonic",
    },

    "Suite 02b — Monotonic Magnitude Decay (50)": {
        "anchor": "50 items",
        "candidates": [
            "49 items", "51 items",
            "45 items", "55 items",
            "40 items", "60 items",
            "30 items", "70 items",
            "10 items", "90 items",
            "5 items",  "200 items",
        ],
        "expected_order": ["49 items", "51 items", "45 items", "55 items",
                           "40 items", "60 items", "30 items", "70 items",
                           "10 items", "90 items", "5 items", "200 items"],
        "description": "Monotonic decay from anchor=50",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 3 — Cross Magnitude Ordering
    # Tests: does model correctly order across magnitude boundaries?
    # ──────────────────────────────────────────
    "Suite 03 — Cross Magnitude Ordering": {
        "anchor": "100 kg",
        "candidates": [
            "105 kg",    # same magnitude, close
            "90 kg",     # same magnitude, close
            "150 kg",    # same magnitude, medium
            "50 kg",     # same magnitude, medium
            "200 kg",    # one step up
            "20 kg",     # one step down
            "500 kg",    # two steps up
            "10 kg",     # two steps down
            "1000 kg",   # order of magnitude up
            "1 kg",      # order of magnitude down
        ],
        "expected_order": ["105 kg", "90 kg", "150 kg", "50 kg", "200 kg",
                           "20 kg", "500 kg", "10 kg", "1000 kg", "1 kg"],
        "description": "Ordering should hold cleanly across magnitude boundaries",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 4 — Small Number Precision
    # Tests: fine-grained ordering for small numbers
    # ──────────────────────────────────────────
    "Suite 04 — Small Number Precision": {
        "anchor": "5 items",
        "candidates": [
            "6 items", "4 items",
            "7 items", "3 items",
            "8 items", "2 items",
            "10 items", "1 item",
            "15 items",
            "50 items",
        ],
        "expected_order": ["6 items", "4 items", "7 items", "3 items", "8 items",
                           "2 items", "10 items", "1 item", "15 items", "50 items"],
        "description": "Small number fine-grained ordering — ±1 then ±2 etc",
        "metric": "monotonic",
    },

    "Suite 04b — Small Number vs Large Number": {
        "anchor": "13 items",
        "candidates": [
            "14 items",   # distance 1
            "12 items",   # distance 1
            "20 items",   # distance 7
            "10 items",   # distance 3
            "45 items",   # distance 32
            "100 items",  # distance 87
            "123 items",  # distance 110 — should rank BELOW 45
            "500 items",  # very far
        ],
        "expected_order": ["14 items", "12 items", "10 items", "20 items",
                           "45 items", "100 items", "123 items", "500 items"],
        "description": "45 must rank above 123 for anchor=13 (common failure case)",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 5 — Large Number Precision
    # Tests: fine-grained ordering for large numbers
    # ──────────────────────────────────────────
    "Suite 05 — Large Number Precision": {
        "anchor": "10000 units",
        "candidates": [
            "10001 units", "9999 units",
            "10010 units", "9990 units",
            "10100 units", "9900 units",
            "11000 units", "9000 units",
            "20000 units", "5000 units",
            "100000 units","1000 units",
        ],
        "expected_order": ["10001 units", "9999 units", "10010 units", "9990 units",
                           "10100 units", "9900 units", "11000 units", "9000 units",
                           "20000 units", "5000 units", "100000 units", "1000 units"],
        "description": "Large number fine-grained ordering around 10000",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 6 — Decimal / Fractional Ordering
    # Tests: can model handle decimal proximity?
    # ──────────────────────────────────────────
    "Suite 06 — Decimal Ordering": {
        "anchor": "2.5 liters",
        "candidates": [
            "2.6 liters", "2.4 liters",
            "2.7 liters", "2.3 liters",
            "3.0 liters", "2.0 liters",
            "3.5 liters", "1.5 liters",
            "5.0 liters", "1.0 liter",
            "10.0 liters",
        ],
        "expected_order": ["2.6 liters", "2.4 liters", "2.7 liters", "2.3 liters",
                           "3.0 liters", "2.0 liters", "3.5 liters", "1.5 liters",
                           "5.0 liters", "1.0 liter", "10.0 liters"],
        "description": "Decimal values ordered correctly by proximity",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 7 — Round Number Bias
    # Tests: does model avoid token-pattern shortcuts?
    # Round numbers share clean token patterns — model must not use this as proxy
    # ──────────────────────────────────────────
    "Suite 07 — Round Number Bias (500)": {
        "anchor": "500 meters",
        "candidates": [
            "501 meters",   # ±1 — irregular token, must rank HIGH
            "499 meters",   # ±1 — irregular token, must rank HIGH
            "505 meters",   # ±5
            "495 meters",   # ±5
            "510 meters",   # ±10
            "600 meters",   # ±100 — clean round, must rank BELOW 501/499
            "400 meters",   # ±100 — clean round, must rank BELOW 501/499
            "1000 meters",  # ±500 — very clean round, must rank lowest
        ],
        "expected_order": ["501 meters", "499 meters", "505 meters", "495 meters",
                           "510 meters", "600 meters", "400 meters", "1000 meters"],
        "description": "Irregular neighbors (501, 499) must rank above clean rounds (600, 400)",
        "metric": "top_k",
        "top_k": 2,
        "top_k_expected": ["501 meters", "499 meters"],
    },

    "Suite 07b — Round Number Bias (1000)": {
        "anchor": "1000 units",
        "candidates": [
            "1001 units",   # ±1
            "999 units",    # ±1
            "1010 units",   # ±10
            "990 units",    # ±10
            "2000 units",   # ×2 — very clean round
            "500 units",    # ÷2 — very clean round
            "10000 units",  # ×10 — extremely clean round
            "100 units",    # ÷10
        ],
        "expected_order": ["1001 units", "999 units", "1010 units", "990 units",
                           "2000 units", "500 units", "10000 units", "100 units"],
        "description": "1001/999 must rank above 2000/500 despite token pattern similarity of round numbers",
        "metric": "top_k",
        "top_k": 2,
        "top_k_expected": ["1001 units", "999 units"],
    },

    # ──────────────────────────────────────────
    # Suite 8 — Symmetric Distance
    # Tests: is distance symmetric? sim(a,b) ≈ sim(b,a)?
    # ──────────────────────────────────────────
    "Suite 08 — Symmetric Distance": {
        "anchor": "200 grams",
        "candidates": [
            "210 grams",  # +10
            "190 grams",  # -10 — should ≈ same similarity as 210
            "250 grams",  # +50
            "150 grams",  # -50 — should ≈ same similarity as 250
            "400 grams",  # +200
            "100 grams",  # -100
        ],
        "expected_order": ["210 grams", "190 grams", "250 grams",
                           "150 grams", "400 grams", "100 grams"],
        "description": "Symmetric values (±10, ±50) should have similar similarity scores",
        "metric": "symmetric",
        "symmetric_pairs": [
            ("210 grams", "190 grams"),
            ("250 grams", "150 grams"),
        ],
    },

    # ──────────────────────────────────────────
    # Suite 9 — Order of Magnitude Boundary
    # Tests: does model correctly handle the 9→10, 99→100 boundaries?
    # ──────────────────────────────────────────
    "Suite 09 — Magnitude Boundary (9 to 10)": {
        "anchor": "9 items",
        "candidates": [
            "10 items",   # just over boundary — should be CLOSE
            "8 items",    # just under
            "11 items",   # slightly further over
            "7 items",    # slightly further under
            "15 items",   # medium distance
            "5 items",    # medium distance
            "50 items",   # far
            "1 item",     # far
            "100 items",  # very far
        ],
        "expected_order": ["10 items", "8 items", "11 items", "7 items",
                           "15 items", "5 items", "50 items", "1 item", "100 items"],
        "description": "9→10 magnitude boundary: 10 should rank above 15 even though it crosses mag boundary",
        "metric": "top_k",
        "top_k": 2,
        "top_k_expected": ["10 items", "8 items"],
    },

    "Suite 09b — Magnitude Boundary (99 to 100)": {
        "anchor": "99 kg",
        "candidates": [
            "100 kg",   # just over boundary
            "98 kg",
            "101 kg",
            "97 kg",
            "110 kg",
            "90 kg",
            "150 kg",
            "50 kg",
        ],
        "expected_order": ["100 kg", "98 kg", "101 kg", "97 kg",
                           "110 kg", "90 kg", "150 kg", "50 kg"],
        "description": "99→100 boundary: 100 must rank above 110 even with magnitude change",
        "metric": "top_k",
        "top_k": 2,
        "top_k_expected": ["100 kg", "98 kg"],
    },

    # ──────────────────────────────────────────
    # Suite 10 — Financial Values
    # Tests: domain-relevant ordering (your training data domain)
    # ──────────────────────────────────────────
    "Suite 10 — Financial Values": {
        "anchor": "revenue of $250M",
        "candidates": [
            "revenue of $255M",
            "revenue of $245M",
            "revenue of $260M",
            "revenue of $240M",
            "revenue of $300M",
            "revenue of $200M",
            "revenue of $500M",
            "revenue of $100M",
            "revenue of $1B",
            "revenue of $50M",
        ],
        "expected_order": ["revenue of $255M", "revenue of $245M", "revenue of $260M",
                           "revenue of $240M", "revenue of $300M", "revenue of $200M",
                           "revenue of $500M", "revenue of $100M", "revenue of $1B",
                           "revenue of $50M"],
        "description": "Financial context: numeric ordering holds with $ and M suffix",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 11 — Percentage Values
    # ──────────────────────────────────────────
    "Suite 11 — Percentage Values": {
        "anchor": "growth of 10%",
        "candidates": [
            "growth of 11%",
            "growth of 9%",
            "growth of 12%",
            "growth of 8%",
            "growth of 15%",
            "growth of 5%",
            "growth of 20%",
            "growth of 2%",
            "growth of 50%",
            "growth of 0%",
        ],
        "expected_order": ["growth of 11%", "growth of 9%", "growth of 12%", "growth of 8%",
                           "growth of 15%", "growth of 5%", "growth of 20%", "growth of 2%",
                           "growth of 50%", "growth of 0%"],
        "description": "Percentage ordering — common in financial text",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 12 — Zero / Near-Zero Anchors
    # Tests: behavior around zero (your 0% growth case)
    # ──────────────────────────────────────────
    "Suite 12 — Near Zero Ordering": {
        "anchor": "change of 0%",
        "candidates": [
            "change of 1%",
            "change of 2%",
            "change of 5%",
            "change of 10%",
            "change of 20%",
            "change of 50%",
        ],
        "expected_order": ["change of 1%", "change of 2%", "change of 5%",
                           "change of 10%", "change of 20%", "change of 50%"],
        "description": "From zero anchor, similarity should decay as value increases",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 13 — Scientific / Very Large Scale
    # ──────────────────────────────────────────
    "Suite 13 — Large Scale Ordering": {
        "anchor": "market cap of $50B",
        "candidates": [
            "market cap of $51B",
            "market cap of $49B",
            "market cap of $55B",
            "market cap of $45B",
            "market cap of $60B",
            "market cap of $40B",
            "market cap of $75B",
            "market cap of $25B",
            "market cap of $100B",
            "market cap of $10B",
        ],
        "expected_order": ["market cap of $51B", "market cap of $49B", "market cap of $55B",
                           "market cap of $45B", "market cap of $60B", "market cap of $40B",
                           "market cap of $75B", "market cap of $25B", "market cap of $100B",
                           "market cap of $10B"],
        "description": "Very large numbers — ordering holds at billion scale",
        "metric": "monotonic",
    },

    # ──────────────────────────────────────────
    # Suite 14 — Context Invariance
    # Tests: same numbers, different sentence contexts → same ordering
    # ──────────────────────────────────────────
    "Suite 14 — Context Invariance (bare numbers)": {
        "anchor": "321",
        "candidates": ["323", "319", "325", "317", "330", "310",
                       "350", "300", "400", "250", "500", "100"],
        "expected_order": ["323", "319", "325", "317", "330", "310",
                           "350", "300", "400", "250", "500", "100"],
        "description": "Bare numbers with no unit context — pure numeric ordering",
        "metric": "monotonic",
    },

    "Suite 14b — Context Invariance (sentence context)": {
        "anchor": "the company reported 321 employees",
        "candidates": [
            "the company reported 323 employees",
            "the company reported 319 employees",
            "the company reported 330 employees",
            "the company reported 310 employees",
            "the company reported 400 employees",
            "the company reported 250 employees",
            "the company reported 500 employees",
            "the company reported 100 employees",
        ],
        "expected_order": [
            "the company reported 323 employees",
            "the company reported 319 employees",
            "the company reported 330 employees",
            "the company reported 310 employees",
            "the company reported 400 employees",
            "the company reported 250 employees",
            "the company reported 500 employees",
            "the company reported 100 employees",
        ],
        "description": "Same numbers embedded in sentence context — ordering should hold",
        "metric": "monotonic",
    },

}


In [17]:
# Run all suites
for name, suite in TEST_SUITES.items():
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'  {suite["description"]}')
    print(f' {suite["expected_order"]}')
    probe(suite['anchor'], suite['candidates'])



Suite 01 — Immediate Neighbor Recognition (small)
  Immediate neighbors (±1) should rank above distant values
 ['14 ml', '12 ml', '15 ml', '11 ml', '20 ml', '10 ml', '25 ml', '5 ml', '30 ml', '3 ml', '45 ml', '100 ml', '321 ml']

Anchor: 13 ml 
Similarity ranking (highest → lowest):
       14 ml : 0.9913 
       12 ml : 0.9899 
       11 ml : 0.9832 
       15 ml : 0.9828 
       10 ml : 0.9757 
       20 ml : 0.9654 
       25 ml : 0.9594 
       30 ml : 0.9515 
       45 ml : 0.9288 
        5 ml : 0.9270 
        3 ml : 0.9232 
      100 ml : 0.9181 
      321 ml : 0.8786 

  Numeric distance correlation: -0.7534
  (≤ -0.7 = learned numeric ordering | ≈ 0 = no magnitude structure)

Suite 01b — Immediate Neighbor Recognition (large)
  ±1 neighbors rank above ±100 rank above order-of-magnitude jumps
 ['1001 units', '999 units', '1005 units', '995 units', '1010 units', '990 units', '1050 units', '950 units', '1100 units', '900 units', '2000 units', '500 units']

Anchor: 1000 units 
Si

In [ ]:
test_0 = {"Suite 02 — Monotonic Magnitude Decay (500)": {
        "anchor": "500 dollars",
        "candidates": [
            "499 dollars", "501 dollars",     # ±1
            "490 dollars", "510 dollars",     # ±10
            "450 dollars", "550 dollars",     # ±50
            "400 dollars", "600 dollars",     # ±100
            "300 dollars", "700 dollars",     # ±200
            "100 dollars", "900 dollars",     # ±400
        ],
        "expected_order": ["499 dollars", "501 dollars", "490 dollars", "510 dollars", "450 dollars",
                           "550 dollars", "400 dollars", "600 dollars", "300 dollars", "700 dollars",
                           "100 dollars", "900 dollars"],
        "description": "Similarity should decay monotonically as numeric distance grows from 500",
        "metric": "monotonic",
    }
}
for name, suite in test_0.items():
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'  {suite["description"]}')
    probe(suite['anchor'], suite['candidates'])


Suite 02 — Monotonic Magnitude Decay (500)
  Similarity should decay monotonically as numeric distance grows from 500

Anchor: 500 dollars 
Similarity ranking (highest → lowest):
  400 dollars : 0.9880 
  600 dollars : 0.9839 
  700 dollars : 0.9785 
  300 dollars : 0.9780 
  501 dollars : 0.9744 
  499 dollars : 0.9732 
  510 dollars : 0.9730 
  900 dollars : 0.9719 
  490 dollars : 0.9629 
  100 dollars : 0.9619 
  550 dollars : 0.9593 
  450 dollars : 0.9589 

  Numeric distance correlation: -0.0101
  (≤ -0.7 = learned numeric ordering | ≈ 0 = no magnitude structure)


In [ ]:
nchor: 500 ml  ['500', 'Ġml']
Similarity ranking (highest → lowest):
      400 ml : 0.9863   ← ['400', 'Ġml']
      600 ml : 0.9855   ← ['600', 'Ġml']
      510 ml : 0.9845   ← ['510', 'Ġml']
      700 ml : 0.9833   ← ['700', 'Ġml']
      499 ml : 0.9828   ← ['499', 'Ġml']
      550 ml : 0.9826   ← ['550', 'Ġml']
      501 ml : 0.9817   ← ['501', 'Ġml']
      490 ml : 0.9815   ← ['490', 'Ġml']
      300 ml : 0.9782   ← ['300', 'Ġml']
      450 ml : 0.9758   ← ['450', 'Ġml']
      900 ml : 0.9633   ← ['900', 'Ġml']
      100 ml : 0.9508   ← ['100', 'Ġml']

In [ ]:
def improved_numeric_loss(
    anchor_emb,
    pos_emb,
    neg_emb,
    anchor_score,
    pos_score,
    neg_score,
    anchor_value,
    pos_value,
    neg_value,
    base_margin=0.2,
    alpha=0.5,
    beta=0.6,
    eps=1e-8,
):
    """
    Improved version with:
    - Non-saturating log scaling
    - Stronger dynamic margin
    - Global pairwise alignment
    """

    # --------------------------------------------------
    # 1. Normalize embeddings (metric space only)
    # --------------------------------------------------
    anchor = F.normalize(anchor_emb, dim=-1)
    pos    = F.normalize(pos_emb, dim=-1)
    neg    = F.normalize(neg_emb, dim=-1)

    # --------------------------------------------------
    # 2. Cosine distances ∈ [0, 2]
    # --------------------------------------------------
    pos_cos = 1.0 - F.cosine_similarity(anchor, pos, dim=-1)
    neg_cos = 1.0 - F.cosine_similarity(anchor, neg, dim=-1)
    pn_cos  = 1.0 - F.cosine_similarity(pos, neg, dim=-1)

    # --------------------------------------------------
    # 3. Log-space numeric distances
    # --------------------------------------------------
    log_a = torch.log1p(anchor_value + eps)
    log_p = torch.log1p(pos_value + eps)
    log_n = torch.log1p(neg_value + eps)

    log_pos = torch.abs(log_a - log_p)
    log_neg = torch.abs(log_a - log_n)
    log_pn  = torch.abs(log_p - log_n)

    # --------------------------------------------------
    # 4. Stronger Dynamic Triplet
    # --------------------------------------------------
    log_diff = log_neg - log_pos
    dyn_margin = base_margin * (1.0 + log_diff.clamp(min=0))

    triplet_loss = F.relu(pos_cos - neg_cos + dyn_margin).mean()

    # --------------------------------------------------
    # 5. Non-saturating Log-Distance Alignment
    #
    # scaling(x) = x / (1 + x)
    # --------------------------------------------------
    def scale(x):
        return x / (1.0 + x)

    norm_pos_cos = pos_cos / 2.0
    norm_neg_cos = neg_cos / 2.0
    norm_pn_cos  = pn_cos  / 2.0

    target_pos = scale(log_pos)
    target_neg = scale(log_neg)
    target_pn  = scale(log_pn)

    log_distance_loss = (
        F.mse_loss(norm_pos_cos, target_pos) +
        F.mse_loss(norm_neg_cos, target_neg) +
        F.mse_loss(norm_pn_cos,  target_pn)
    ) / 3.0

    # --------------------------------------------------
    # 6. Head Regression (unchanged)
    # --------------------------------------------------
    head_loss = (
        F.mse_loss(anchor_score, log_a) +
        F.mse_loss(pos_score,    log_p) +
        F.mse_loss(neg_score,    log_n)
    ) / 3.0

    # --------------------------------------------------
    # 7. Smooth Ranking Loss (better than sign-based)
    #
    # Uses softplus for stability.
    # --------------------------------------------------
    margin_rank = 0.3

    ap = anchor_score - pos_score
    an = anchor_score - neg_score

    ap_sign = torch.sign(log_a - log_p)
    an_sign = torch.sign(log_a - log_n)

    rank_loss = (
        F.softplus(margin_rank - ap_sign * ap).mean() +
        F.softplus(margin_rank - an_sign * an).mean()
    ) / 2.0

    # --------------------------------------------------
    # 8. Weighted Grouping
    # --------------------------------------------------
    metric_loss = triplet_loss + log_distance_loss
    supervision_loss = beta * head_loss + (1 - beta) * rank_loss

    total_loss = alpha * metric_loss + (1 - alpha) * supervision_loss

    components = {
        "triplet": triplet_loss.item(),
        "log_dist": log_distance_loss.item(),
        "head": head_loss.item(),
        "rank": rank_loss.item(),
        "metric": metric_loss.item(),
        "supervision": supervision_loss.item(),
        "total": total_loss.item(),
    }

    return total_loss, components

In [ ]:
class ContrastiveModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.numeric_head = nn.Linear(hidden_size, 1)  # sees raw emb
        # Optional: separate projection for cosine space
        self.metric_proj = nn.Linear(hidden_size, hidden_size)  # sees raw, outputs normalized

    def encode(self, batch_part):
        out = self.encoder(
            input_ids=batch_part["input_ids"],
            attention_mask=batch_part["attention_mask"],
        )
        return mean_pooling(out, batch_part["attention_mask"])  # raw

    def forward(self, batch):
        a_emb = self.encode(batch["anchor"])
        p_emb = self.encode(batch["positive"])
        n_emb = self.encode(batch["negative"])

        # Head sees raw — preserves magnitude signal
        a_score = self.numeric_head(a_emb).squeeze(-1)
        p_score = self.numeric_head(p_emb).squeeze(-1)
        n_score = self.numeric_head(n_emb).squeeze(-1)

        # Cosine loss sees projected + normalized — clean directional space
        a_proj = F.normalize(self.metric_proj(a_emb), dim=-1)
        p_proj = F.normalize(self.metric_proj(p_emb), dim=-1)
        n_proj = F.normalize(self.metric_proj(n_emb), dim=-1)

        return {
            "a_emb": a_proj,    # for triplet + log-distance loss
            "p_emb": p_proj,
            "n_emb": n_proj,
            "a_score": a_score, # for head + rank loss
            "p_score": p_score,
            "n_score": n_score,
        }